# Phase 5: Reinforcement Learning via GRPO

Refine the SFT model's tool-calling behavior with **Group Relative Policy Optimization (GRPO)**.

**Why RL after SFT?** SFT teaches the model *what good tool-calling looks like* by imitating the teacher (GPT-5.4). But imitation has a ceiling — the student copies the teacher's patterns without understanding *why* they work. GRPO lets the model explore alternative strategies and learn from outcomes: "calling `get_financials` before `get_price_history` leads to higher-quality memos" is the kind of signal RL can discover that SFT cannot.

**What is GRPO?** DeepSeek's alternative to PPO. Instead of training a separate critic/value model, GRPO generates *groups* of completions per prompt and uses the group's relative scores as the baseline. This eliminates ~50% of memory overhead (no reference model when `beta=0.0`).

**Pipeline position:** Phase 4 (SFT) checkpoint --> **Phase 5 (GRPO)** --> final model

## The Multi-Turn Problem

Our task is inherently **multi-turn**: the model thinks, calls a tool, receives results, thinks again, calls another tool, and eventually produces a final research memo. A full trajectory looks like:

```
system --> user --> assistant (think + tool_call) --> tool_result --> assistant (think + tool_call) --> tool_result --> ... --> assistant (final memo)
```

This creates a fundamental tension with standard RL frameworks:

### TRL GRPOTrainer: Single-Turn Only
TRL's `GRPOTrainer` generates one completion per prompt and scores it. It has no mechanism to:
- Pause generation when a tool call is emitted
- Execute the tool and inject the result back into context
- Resume generation from the extended context
- Repeat this loop for multiple tool-calling turns

The trainer treats generation as a single `model.generate()` call. There is no environment loop.

### ART (OpenPipe): Native Multi-Turn Support
ART (`openpipe-art`) was built specifically for agentic RL. Its `rollout()` function is an async environment loop where:
- The model generates a response (which may include tool calls)
- Tools are executed and results are appended
- The model generates again, seeing the full history
- This repeats until the task is complete
- The entire trajectory is scored and used for training

The `art-mcp_rl.ipynb` reference notebook demonstrates exactly this pattern — training Qwen 2.5 3B to use MCP tools via GRPO with multi-turn rollouts.

### Our Decision: TRL for Single-Turn Scoring

For this demo, we use **TRL's GRPOTrainer** with a pragmatic simplification:
- **We score the model's first response only** — does it think before acting? Does it call valid tools with correct JSON? Does it select the right tools for the research task?
- The reward function evaluates the *quality of the plan*, not the full trajectory outcome
- This is sufficient to teach tool-calling discipline (valid JSON, correct tool names, thinking before acting)

For **full multi-turn RL** (scoring based on the quality of the final memo after all tools have been called), you would need ART. That is a GPU-intensive process requiring vLLM for fast generation — not something we can demo on a free Colab T4 in 10 minutes. But the reward function design and hyperparameter choices shown here transfer directly.

**Prerequisites:**
- Phase 4 SFT checkpoint (LoRA adapters from Colab training)
- `trajectories_sft.jsonl` from Phase 2 (we extract prompts + ground truth from it)
- Colab runtime with T4 GPU

## 1. Install Dependencies

Same stack as Phase 4 SFT, plus `trl>=0.22.2` for GRPOTrainer.

Qwen3.5 needs `flash-linear-attention` + `causal_conv1d` for its hybrid DeltaNet architecture.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
%%capture
# Unsloth + Qwen3.5 dependencies (takes ~3-5 min first time)
!pip install -qqq \
    "torch==2.8.0" "triton>=3.3.0" numpy pillow torchvision bitsandbytes xformers==0.0.32.post2 \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!pip install transformers==5.2.0
!pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [ ]:
# Restart runtime after install (required for Unsloth to patch correctly)
import os
os.kill(os.getpid(), 9)

In [ ]:
# After restart: import unsloth FIRST (before torch) to apply optimizations
import unsloth
import torch
print(f"Unsloth: {unsloth.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB free / {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB total")

## 2. Load the SFT Model

Load the Phase 4 checkpoint. Two options:
- **LoRA adapters** from `qwen35_2b_sft_lora/` (smaller upload, loads on top of base model)
- **HuggingFace Hub** if you pushed the LoRA adapters in Phase 4

For GRPO on Qwen3.5, `fast_inference=False` is required — vLLM does not yet support this architecture.
Unsloth's native inference handles generation during the GRPO rollouts instead.

In [ ]:
from unsloth import FastVisionModel

# Option A: Load from uploaded LoRA adapters
SFT_LORA_PATH = "qwen35_2b_sft_lora"  # upload this directory from Phase 4

# Option B: Load from HuggingFace Hub (if you pushed in Phase 4)
# SFT_LORA_PATH = "your-username/qwen35-2b-tool-calling-sft"

MAX_SEQ_LENGTH = 8192

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=SFT_LORA_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=False,  # Required for Qwen3.5 GRPO — vLLM does not support this arch yet
)

print(f"\nSFT model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. Dataset Setup

GRPO needs a different data format than SFT:
- **SFT:** `prompt` + `completion` (the model learns to reproduce the completion)
- **GRPO:** `prompt` + `ground_truth` (the model generates its own completions, scored against ground truth)

We extract prompts from our SFT trajectories and pair them with the tools the teacher used as ground truth.
The model will generate its own first-turn response and get scored on tool-calling quality.

In [ ]:
import json
import random
from datasets import Dataset
from google.colab import files

# Upload the same trajectories_sft.jsonl from Phase 2
print("Select trajectories_sft.jsonl from your local data/bbb/ directory:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
records = [json.loads(line) for line in uploaded[filename].decode().strip().split("\n")]
print(f"Loaded {len(records)} trajectories")

In [ ]:
# Valid tool names — used by the reward function to check for hallucinated tools
VALID_TOOL_NAMES = {"get_stock_news", "get_financials", "get_price_history", "get_recommendations"}


def extract_grpo_example(record: dict) -> dict | None:
    """
    Convert an SFT trajectory into a GRPO training example.

    Extract:
      - prompt: system + user messages formatted with the chat template (including tool definitions)
      - ground_truth: JSON string listing which tools the teacher called (for reward scoring)
    """
    messages = record["messages"]
    tools = record["tools"]

    # Find system and user messages (the prompt)
    system_msg = next((m for m in messages if m["role"] == "system"), None)
    user_msg = next((m for m in messages if m["role"] == "user"), None)
    if not system_msg or not user_msg:
        return None

    # Extract which tools the teacher actually called (ground truth for scoring)
    teacher_tools_called = []
    for m in messages:
        if m.get("tool_calls"):
            for tc in m["tool_calls"]:
                fn_name = tc.get("function", {}).get("name", "")
                if fn_name and fn_name not in teacher_tools_called:
                    teacher_tools_called.append(fn_name)

    # Build the prompt using chat template — includes tool definitions in the system message
    prompt_messages = [system_msg, user_msg]
    prompt_text = tokenizer.tokenizer.apply_chat_template(
        prompt_messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "prompt": prompt_text,
        "ground_truth": json.dumps({
            "ticker": record.get("ticker", ""),
            "tools_called": teacher_tools_called,
        }),
    }


# Convert all trajectories
grpo_examples = [extract_grpo_example(r) for r in records]
grpo_examples = [e for e in grpo_examples if e is not None]

# Shuffle and split
random.seed(42)
random.shuffle(grpo_examples)

train_dataset = Dataset.from_list(grpo_examples)

print(f"GRPO dataset: {len(train_dataset)} examples")
print(f"\nSample prompt (first 300 chars):\n{'='*60}")
print(train_dataset[0]["prompt"][:300])
print(f"\nGround truth: {train_dataset[0]['ground_truth']}")

## 4. Reward Function

The reward function is the core of RL — it defines what "good" means. Our composite reward scores
the model's single-turn response across multiple dimensions:

| Component | Score | What it measures |
|-----------|-------|-----------------|
| `valid_json` | +1.0 | All `<tool_call>` blocks contain parseable JSON |
| `thinking` | +1.0 | Model uses `<think>` before calling tools |
| `tool_selection` | 0 to +2.0 | Jaccard overlap between called tools and teacher's tool set |
| `no_hallucination` | -2.0 | Penalize calling tools that don't exist |
| `formatting` | +1.0 | Correct `<tool_call>` tag structure |

**Design principles from the WinterFest experiment:**
- **Dense rewards > sparse** — the model always gets partial credit, never all-or-nothing
- **Multiple reward scales** — ensures variance across completions in a group (if all 4 completions score identically, the GRPO gradient is zero and training stalls)
- **Negative penalties for bad behavior** — hallucinated tools get -2.0, much stronger than positive signals

The full `compute_reward()` function lives in `helpers__inference.py`. Here we wrap it to match TRL's expected signature.

In [ ]:
import re

# ---------------------------------------------------------------------------
# Tool call parsing (from helpers__inference.py)
# ---------------------------------------------------------------------------

_TOOL_CALL_PATTERN = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)


def parse_tool_calls_from_text(text: str) -> list[dict]:
    """Extract and parse <tool_call> blocks from raw model output."""
    matches = _TOOL_CALL_PATTERN.findall(text)
    tool_calls = []
    for raw in matches:
        try:
            parsed = json.loads(raw)
            tool_calls.append({
                "name": parsed.get("name", ""),
                "arguments": parsed.get("arguments", {}),
                "valid_json": True,
            })
        except (json.JSONDecodeError, KeyError):
            tool_calls.append({"name": "__malformed__", "arguments": raw, "valid_json": False})
    return tool_calls


# ---------------------------------------------------------------------------
# Reward function — TRL signature: fn(completions, **kwargs) -> list[float]
# ---------------------------------------------------------------------------

def tool_calling_reward_func(completions: list[str], ground_truth: list[str], **kwargs) -> list[float]:
    """
    Score each completion on tool-calling quality.

    TRL's GRPOTrainer calls this with:
      - completions: list of generated strings (one per completion in the group)
      - ground_truth: list of JSON strings with teacher's tool choices (passed via dataset column)

    Returns a list of float rewards, one per completion.
    """
    rewards = []

    for completion, gt_json in zip(completions, ground_truth):
        score = 0.0

        # Handle list-of-dicts format from TRL (content blocks)
        if isinstance(completion, list):
            completion = completion[0].get("content", "") if completion else ""

        # Parse ground truth
        try:
            gt = json.loads(gt_json)
            gt_tools = set(gt.get("tools_called", []))
        except (json.JSONDecodeError, TypeError):
            gt_tools = VALID_TOOL_NAMES  # fallback: expect all tools

        # Parse tool calls from the completion
        tool_calls = parse_tool_calls_from_text(completion)
        called_names = {tc["name"] for tc in tool_calls if tc["name"] != "__malformed__"}

        # --- Component 1: Valid JSON (+1.0) ---
        if tool_calls and all(tc["valid_json"] for tc in tool_calls):
            score += 1.0

        # --- Component 2: Thinking before acting (+1.0) ---
        if "<think>" in completion:
            score += 1.0

        # --- Component 3: Tool selection overlap (0 to +2.0) ---
        if gt_tools and called_names:
            jaccard = len(called_names & gt_tools) / len(called_names | gt_tools)
            score += round(jaccard * 2.0, 2)

        # --- Component 4: No hallucinated tools (-2.0) ---
        hallucinated = called_names - VALID_TOOL_NAMES
        if hallucinated:
            score -= 2.0

        # --- Component 5: Proper formatting (+1.0) ---
        # Check that tool calls use the <tool_call> tags (not raw JSON)
        if tool_calls and not any(tc["name"] == "__malformed__" for tc in tool_calls):
            score += 1.0

        rewards.append(score)

    return rewards


# Quick test
test_completion = """<think>
I need to research AAPL. Let me gather financials and price data.
</think>
<tool_call>
{"name": "get_financials", "arguments": {"ticker": "AAPL", "statement_type": "income", "period": "annual"}}
</tool_call>
<tool_call>
{"name": "get_price_history", "arguments": {"ticker": "AAPL", "period": "6mo", "interval": "1wk"}}
</tool_call>"""

test_gt = json.dumps({"ticker": "AAPL", "tools_called": ["get_financials", "get_price_history", "get_stock_news"]})
test_reward = tool_calling_reward_func([test_completion], [test_gt])
print(f"Test reward: {test_reward[0]:.2f}")
print("  Expected: valid_json(+1) + thinking(+1) + tool_selection(~1.33) + no_hallucination(0) + formatting(+1) = ~4.33")

## 5. GRPOTrainer Configuration

These hyperparameters come from the WinterFest GRPO experiment (1x H100, 12 hours, Qwen3-VL-4B).
Key choices explained:

| Parameter | Value | Why |
|-----------|-------|-----|
| `learning_rate=5e-6` | Much lower than SFT (2e-4) | RL is sensitive — large updates cause catastrophic forgetting of SFT knowledge |
| `max_grad_norm=0.1` | Tight clipping | RL gradients are noisy; tight clipping prevents spikes |
| `beta=0.0` | No KL penalty | Eliminates the reference model entirely, saving ~50% VRAM. GRPO's big simplification over PPO |
| `loss_type="dr_grpo"` | Dr. GRPO variant | Removes length bias — without this, the model learns to be verbose because longer completions accumulate more token-level reward |
| `scale_rewards=False` | Required by Dr. GRPO | Dr. GRPO handles its own normalization |
| `num_generations=4` | 4 completions per prompt | Each prompt gets 4 different completions. Their relative scores become the training signal. More = better signal but more VRAM |
| `temperature=1.0` | High diversity | GRPO *needs* variance. If all 4 completions are identical (low temp), the relative scores are all the same and the gradient is zero |
| `epsilon=0.2` | PPO-style clipping | Prevents the policy from moving too far from the previous step |
| `remove_unused_columns=False` | **MANDATORY** | Without this, TRL strips the `ground_truth` column before passing it to the reward function, and `tool_calling_reward_func` receives nothing |

### GRPOTrainer Footguns (learned the hard way)

1. **`remove_unused_columns=False`** — TRL's default behavior removes any dataset column not in the model's forward signature. Since `ground_truth` is a custom column for the reward function, it gets silently dropped. Your reward function receives empty kwargs and you get mysterious all-zero rewards.

2. **`FastVisionModel.for_training(model)`** must be called before constructing the trainer — otherwise the model stays in inference mode and gradients don't flow correctly.

3. **`processing_class=tokenizer`**, not `tokenizer=tokenizer` — the old kwarg is deprecated in TRL 0.22+. Using the old name silently falls back to a default tokenizer and generation breaks.

4. **Reward function signature** must be `fn(completions: list[str], **kwargs) -> list[float]` — dataset columns arrive via `**kwargs`, not as positional arguments. If your function signature has `ground_truth` as a positional arg, it won't receive it.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# CRITICAL: put model in training mode BEFORE constructing the trainer
FastVisionModel.for_training(model)

training_args = GRPOConfig(
    # --- RL-specific hyperparameters ---
    learning_rate=5e-6,               # 40x lower than SFT — RL is sensitive
    max_grad_norm=0.1,                # tight clipping for noisy RL gradients
    beta=0.0,                         # no KL penalty, no reference model — saves 50% VRAM
    loss_type="dr_grpo",              # Dr. GRPO: no length bias
    scale_rewards=False,              # required by Dr. GRPO
    num_generations=4,                # completions per prompt (the "group" in GRPO)
    temperature=1.0,                  # high diversity — GRPO needs variance
    top_p=1.0,
    top_k=0,
    epsilon=0.2,                      # PPO-style clipping

    # --- Sequence lengths ---
    max_prompt_length=2048,           # tool definitions + system prompt + user message
    max_completion_length=1024,       # first-turn response (think + tool calls)

    # --- Batch sizes ---
    per_device_train_batch_size=1,    # T4 memory constraint
    gradient_accumulation_steps=4,    # effective: 1 * 4 * 4 = 16 completions per update
    # Note: per_device_train_batch_size * gradient_accumulation_steps must be
    # a multiple of num_generations. Unsloth will auto-adjust if needed.

    # --- Training schedule ---
    max_steps=60,                     # short demo run — real training would be 300-500 steps
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.1,
    optim="adamw_8bit",

    # --- Logging ---
    logging_steps=1,
    logging_first_step=True,
    log_completions=True,             # prints sample completions — great for debugging
    num_completions_to_print=1,
    save_steps=30,
    save_total_limit=2,
    report_to="none",
    output_dir="outputs_grpo",

    # --- MANDATORY footgun prevention ---
    remove_unused_columns=False,      # DO NOT REMOVE — keeps ground_truth column for reward func
    importance_sampling_level="sequence",
    mask_truncated_completions=False,

    seed=3407,
)

print("GRPOConfig created successfully")

## 6. Training

The trainer will:
1. For each prompt, generate `num_generations=4` completions at `temperature=1.0`
2. Score each completion with `tool_calling_reward_func`
3. Compute relative advantages within the group (GRPO's core insight)
4. Update the model to increase probability of high-scoring completions

**What to watch for in the logs:**
- `reward` should trend upward over steps
- `reward_std > 0` every step — if it hits 0, all completions scored identically and the gradient is zero (training stalls)
- `kl = 0.0` always (expected with `beta=0.0`)
- Sample completions should show `<think>` blocks and valid `<tool_call>` tags

In [ ]:
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,       # NOT tokenizer=tokenizer — that's deprecated
    reward_funcs=[tool_calling_reward_func],
    train_dataset=train_dataset,
)

trainer.train()

## 7. Save GRPO Checkpoint

In [ ]:
# Save LoRA adapters (GRPO updates on top of SFT)
model.save_pretrained("qwen35_2b_grpo_lora")
tokenizer.save_pretrained("qwen35_2b_grpo_lora")
print("GRPO LoRA adapters saved to qwen35_2b_grpo_lora/")

# Optional: save merged model for MLX conversion
# model.save_pretrained_merged("qwen35_2b_grpo_merged", tokenizer)

# Optional: push to HuggingFace Hub
# model.push_to_hub_merged("your-username/qwen35-2b-tool-calling-grpo", tokenizer)

## 8. Evaluation — SFT vs GRPO

Run the same prompts through the GRPO model and compare rewards against the SFT baseline.
This is single-turn evaluation (first response only) since we are scoring the same thing we trained on.

In [ ]:
FastVisionModel.for_inference(model)

# Pick a few test tickers
eval_tickers = ["MSFT", "NVDA", "JNJ", "XOM", "COST"]
eval_examples = [e for e in grpo_examples if any(t in e["ground_truth"] for t in eval_tickers)][:5]

if not eval_examples:
    eval_examples = grpo_examples[:5]  # fallback

print(f"Evaluating {len(eval_examples)} examples\n")
print(f"{'Ticker':<8} {'Reward':>8} {'Think':>6} {'Tools Called':<40} {'Valid JSON':>10}")
print("=" * 80)

for ex in eval_examples:
    gt = json.loads(ex["ground_truth"])
    ticker = gt.get("ticker", "?")

    # Tokenize and generate
    inputs = tokenizer.tokenizer(ex["prompt"], return_tensors="pt").input_ids.to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.6,        # lower temp for eval (deterministic-ish)
        top_p=0.95,
        do_sample=True,
    )
    response = tokenizer.tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=False)

    # Clean up special tokens
    for tok in ["<|im_end|>", "<|endoftext|>", "<|im_start|>"]:
        response = response.replace(tok, "")

    # Score
    reward = tool_calling_reward_func([response], [ex["ground_truth"]])[0]
    tool_calls = parse_tool_calls_from_text(response)
    called = [tc["name"] for tc in tool_calls]
    has_think = "<think>" in response
    all_valid = all(tc["valid_json"] for tc in tool_calls) if tool_calls else False

    print(f"{ticker:<8} {reward:>8.2f} {'Y' if has_think else 'N':>6} {', '.join(called) or '(none)':<40} {'Y' if all_valid else 'N':>10}")

print(f"\n{'='*80}")
print("Compare these rewards against SFT baseline from Phase 4 evaluation.")
print("Improvement in reward = GRPO is learning better tool-calling strategies.")

## 9. What We Trained vs What We Want

### What this notebook does (single-turn GRPO via TRL)
- Scores the model's **first response** to a research prompt
- Rewards: valid JSON tool calls, thinking before acting, correct tool selection, no hallucinations
- This teaches **tool-calling discipline** — the mechanical skills of producing well-formed tool calls

### What full multi-turn RL would do (ART / custom environment)
- Runs the complete agent loop: think -> call tool -> receive result -> think -> call tool -> ... -> final memo
- Scores the **entire trajectory** based on the quality of the final research memo
- This teaches **research strategy** — which tools to call in what order, how to synthesize results

### The gap
Single-turn RL cannot teach the model to:
- Adapt its strategy based on tool results (e.g., "financials look weak, better check news for context")
- Know when it has enough data to write the memo vs when to gather more
- Produce a high-quality final synthesis

For full multi-turn RL, you would use **ART** (OpenPipe):
```python
# ART multi-turn RL — conceptual sketch
import art
from art.local import LocalBackend

model = art.TrainableModel(name="qwen35-tool-caller", base_model="path/to/sft/checkpoint")
backend = LocalBackend()
await model.register(backend)

async def rollout(model, scenario):
    traj = art.Trajectory(messages_and_choices=[...])
    # Run the full agent loop with real tool execution
    for turn in range(max_turns):
        response = await client.chat.completions.create(...)
        traj.messages_and_choices.append(response.choices[0])
        if has_tool_calls(response):
            results = execute_tools(response)
            traj.messages_and_choices.append(tool_result_msg)
        else:
            break
    # Score the entire trajectory
    traj.reward = score_final_memo(traj)
    return traj

# ART handles the RL training loop internally
groups = await art.gather_trajectory_groups(...)
await model.train(groups, config=art.TrainConfig(learning_rate=5e-6))
```

ART's `rollout()` is the key difference — it is an async environment loop where tools are actually executed between model turns. TRL's GRPOTrainer has no equivalent; it treats generation as a single `model.generate()` call.

### Practical takeaway for the audience
- **SFT** teaches format and basic competence (Phase 4)
- **Single-turn GRPO** sharpens tool-calling mechanics (this notebook)
- **Multi-turn GRPO via ART** teaches end-to-end research strategy (future work, needs GPU)
- Each layer builds on the last. You do not need multi-turn RL to get a useful model — SFT + single-turn GRPO already produces a model that calls the right tools with valid arguments.